# T89 Magnetospheric Magnetic Field Model — Implementation / 구현

**Paper**: Tsyganenko, N. A. (1989). A Magnetospheric Magnetic Field Model with a Warped Tail Current Sheet. *Planetary and Space Science*, 37(1), 5–20.

This notebook implements key components of the T89 model:
이 노트북은 T89 모델의 핵심 구성 요소를 구현합니다:

1. **Axially symmetric current disc** — vector potential and magnetic field / 축대칭 전류 디스크 — 벡터 포텐셜과 자기장
2. **Warping function $Z_s$** — tail current sheet warping due to dipole tilt / Warping 함수 — dipole tilt에 의한 전류판 휘어짐
3. **Simplified T89 field model** — dipole + tail + ring current / 간소화된 T89 자기장 모델
4. **Field line tracing** — visualizing magnetospheric structure at different Kp / 자기장 선 추적 — Kp별 자기권 구조 시각화

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# Earth radius in km (for reference; all calculations in R_E units)
R_E = 6371.0  # km

## Part 1: Axially Symmetric Current Disc / 축대칭 전류 디스크

The paper starts by deriving the vector potential for an infinitely thin, axially symmetric current sheet (Section 2). The key solutions $A^{(1)}$, $A^{(2)}$, $A^{(3)}$ (eqs. 7–9) represent current discs with different radial decrease rates.

논문은 무한히 얇은 축대칭 전류판의 벡터 포텐셜 유도로 시작합니다 (Section 2). 핵심 해 $A^{(1)}$, $A^{(2)}$, $A^{(3)}$ (식 7–9)은 서로 다른 방사형 감소율을 가진 전류 디스크를 나타냅니다.

We implement these with finite thickness by replacing $|z|$ with $\sqrt{z^2 + D^2}$.

$|z|$를 $\sqrt{z^2 + D^2}$로 치환하여 유한 두께를 도입합니다.

In [ ]:
def current_disc_potentials(rho, z, a=1.0, D=0.25):
    """Compute the three vector potential solutions A^(1), A^(2), A^(3).

    Args:
        rho: Cylindrical radial distance (R_E).
        z: Height above equatorial plane (R_E).
        a: Radial scale length (R_E).
        D: Half-thickness of the current sheet (R_E).

    Returns:
        A1, A2, A3: Vector potentials for the three solutions.
    """
    # Finite thickness: replace |z| with sqrt(z^2 + D^2)
    zeta = np.sqrt(z**2 + D**2)
    az = a + zeta
    r_term = np.sqrt(az**2 + rho**2)

    # A^(1) ~ rho^{-1} [sqrt((a+|z|)^2 + rho^2) - (a+|z|)]  (eq. 7')
    A1 = np.where(rho > 1e-10, (r_term - az) / rho, 0.0)

    # A^(2) = -dA^(1)/da  (eq. 8)
    A2 = np.where(rho > 1e-10, -rho**(-1) * (az / r_term - 1), 0.0)

    # A^(3) = -dA^(2)/da  (eq. 9)
    A3 = np.where(rho > 1e-10, rho / r_term**3, 0.0)

    return A1, A2, A3


def magnetic_field_from_A(rho, z, A_func, a=1.0, D=0.25):
    """Compute B_rho and B_z from vector potential via numerical differentiation.

    B_rho = -dA/dz,  B_z = (1/rho) * d(rho*A)/drho

    Args:
        rho: Cylindrical radial distance (R_E).
        z: Height above equatorial plane (R_E).
        A_func: Function returning A(rho, z, a, D).
        a: Radial scale length.
        D: Half-thickness.

    Returns:
        B_rho, B_z: Magnetic field components.
    """
    h = 1e-5
    A_zp = A_func(rho, z + h, a, D)
    A_zm = A_func(rho, z - h, a, D)
    B_rho = -(A_zp - A_zm) / (2 * h)

    A_rp = A_func(rho + h, z, a, D)
    A_rm = A_func(rho - h, z, a, D)
    rA_rp = (rho + h) * A_rp
    rA_rm = (rho - h) * A_rm
    B_z = np.where(rho > 1e-10, (rA_rp - rA_rm) / (2 * h * rho), 0.0)

    return B_rho, B_z


# Visualize the three current disc potentials and corresponding B_rho profiles
rho_grid = np.linspace(0.01, 8, 200)
z_val = 0.0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, (D_val, label) in enumerate([(0.0, 'D=0 (infinitely thin)'),
                                     (0.25, 'D=0.25'),
                                     (1.0, 'D=1.0')]):
    A1, A2, A3 = current_disc_potentials(rho_grid, z_val, a=1.0, D=max(D_val, 1e-6))
    ax = axes[i]
    ax.plot(rho_grid, A1, 'b-', label=r'$A^{(1)}$', linewidth=2)
    ax.plot(rho_grid, A2, 'r--', label=r'$A^{(2)}$', linewidth=2)
    ax.plot(rho_grid, A3 * 5, 'g-.', label=r'$A^{(3)} \times 5$', linewidth=2)
    ax.set_xlabel(r'$\rho$ ($R_E$)')
    ax.set_ylabel(r'$A_\phi$')
    ax.set_title(f'{label}')
    ax.legend()
    ax.set_ylim(-0.5, 2.0)
    ax.grid(True, alpha=0.3)

fig.suptitle('Current Disc Vector Potentials (eq. 7–9) / 전류 디스크 벡터 포텐셜',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Reproduce Fig. 1: B_rho profiles in the equatorial plane for different D values
# 그림 1 재현: 서로 다른 D 값에 대한 적도면에서의 B_rho 프로필

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def A1_func(rho, z, a, D):
    zeta = np.sqrt(z**2 + D**2)
    az = a + zeta
    r_term = np.sqrt(az**2 + rho**2)
    return np.where(rho > 1e-10, (r_term - az) / rho, 0.0)

for i, (a_val, D_val) in enumerate([(1.0, 0.0), (1.0, 0.25), (1.0, 1.0)]):
    rho_arr = np.linspace(0.01, 8, 300)
    D_eff = max(D_val, 1e-6)

    # Compute j_phi ~ B_rho(z=0+) from numerical derivative
    z_above = 0.01
    B_rho, B_z = magnetic_field_from_A(rho_arr, z_above, A1_func, a=a_val, D=D_eff)

    ax = axes[i]
    ax.plot(rho_arr, -B_rho, 'b-', linewidth=2, label=r'$j_\phi$ (from $A^{(1)}$)')
    ax.plot(rho_arr, B_z, 'r--', linewidth=2, label=r'$B_z$ (from $A^{(1)}$)')
    ax.set_xlabel(r'$\rho$ ($R_E$)')
    ax.set_title(f'a={a_val}, D={D_val}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.5, 1.5)

fig.suptitle('Current density and $B_z$ profiles in equatorial plane (cf. Fig. 1)\n'
             '적도면에서의 전류 밀도 및 $B_z$ 프로필 (그림 1 참조)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Warping Function $Z_s$ / Warping 함수

The key innovation of T89: the tail current sheet warps away from the dipole equatorial plane as a function of the dipole tilt angle $\psi$ (eq. 11):

T89의 핵심 혁신: tail current sheet가 dipole tilt angle $\psi$의 함수로 dipole 적도면에서 벗어나 휘어집니다 (식 11):

$$Z_s(x, y, \psi) = 0.5 \tan\psi \left[x + R_s - \sqrt{(x + R_s)^2 + 16}\right] \cdot (y^4 + L_y^4)^{-1}$$

- Near Earth ($x \gg R_s$): $Z_s \to 0$ (follows dipole equatorial plane / dipole 적도면 따름)
- Far tail ($x \ll -R_s$): $Z_s \to$ constant (approaches solar magnetospheric equatorial plane / 태양 자기권 적도면에 접근)
- $y$ dependence: maximum warping at midnight ($y=0$), decreasing toward flanks / $y$ 의존성: 자정 방향에서 최대, 옆면으로 감소

In [ ]:
def warping_Zs(x, y, psi_deg, Rs=8.0, G=10.0, Ly=10.0):
    """Compute the warping function Z_s (eq. 11).

    Args:
        x: GSM x-coordinate (R_E). Positive toward Sun.
        y: GSM y-coordinate (R_E).
        psi_deg: Dipole tilt angle (degrees).
        Rs: Hinging distance parameter (R_E).
        G: Transverse bending parameter (R_E).
        Ly: Y-scale length (R_E), fixed at 10 R_E.

    Returns:
        Z_s: Warping displacement (R_E).
    """
    psi = np.radians(psi_deg)
    x_term = x + Rs - np.sqrt((x + Rs)**2 + G**2)
    y_term = y**4 + Ly**4
    return 0.5 * np.tan(psi) * x_term * Ly**4 / y_term


# Reproduce Fig. 2: Warped current sheet geometry for psi=30 degrees
# 그림 2 재현: psi=30도에서의 warped current sheet 기하학

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: XZ cross-section at y=0 (midnight meridian)
x_arr = np.linspace(-40, 15, 300)
for psi_val in [0, 10, 20, 30]:
    Zs = warping_Zs(x_arr, 0.0, psi_val)
    axes[0].plot(x_arr, Zs, linewidth=2, label=fr'$\psi={psi_val}°$')

axes[0].set_xlabel(r'$X_{GSM}$ ($R_E$)')
axes[0].set_ylabel(r'$Z_s$ ($R_E$)')
axes[0].set_title('XZ cross-section at y=0 (midnight meridian)\n'
                   'y=0에서의 XZ 단면 (자정 자오면)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='k', linewidth=0.5)

# Right: YZ cross-section at x=-10 R_E
y_arr = np.linspace(-20, 20, 300)
for psi_val in [0, 10, 20, 30]:
    Zs = warping_Zs(-10.0, y_arr, psi_val)
    axes[1].plot(y_arr, Zs, linewidth=2, label=fr'$\psi={psi_val}°$')

axes[1].set_xlabel(r'$Y_{GSM}$ ($R_E$)')
axes[1].set_ylabel(r'$Z_s$ ($R_E$)')
axes[1].set_title(r'YZ cross-section at $X_{GSM}=-10\,R_E$' '\n'
                   r'$X_{GSM}=-10\,R_E$에서의 YZ 단면')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='k', linewidth=0.5)

fig.suptitle('Warped Tail Current Sheet Geometry (eq. 11, cf. Fig. 2)\n'
             'Warped 꼬리 전류판 기하학', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Simplified T89 Field Model / 간소화된 T89 자기장 모델

We implement a simplified version of the T89 model in the noon-midnight meridian plane ($y=0$). The total field is:

Noon-midnight 자오면($y=0$)에서 간소화된 T89 모델을 구현합니다. 전체 자기장은:

$$\mathbf{B}_{\text{total}} = \mathbf{B}_{\text{dipole}} + \mathbf{B}_{\text{tail}}$$

The dipole field in GSM coordinates (with tilt $\psi=0$):

GSM 좌표에서의 dipole 자기장 ($\psi=0$):

$$B_x = \frac{3xz}{r^5} B_0, \quad B_z = \frac{3z^2 - r^2}{r^5} B_0$$

The tail contribution uses $A^{(3)}$ (the solution with finite magnetic moment) with a truncation factor $W(x)$ and Kp-dependent amplitude.

Tail 기여는 유한 자기 모멘트를 가진 $A^{(3)}$ 해에 truncation factor $W(x)$와 Kp 의존 진폭을 적용합니다.

In [ ]:
# Simplified T89 parameters from Table 1 (approximate values for key Kp bins)
# Table 1에서 추출한 간소화된 T89 매개변수 (주요 Kp bin의 근사값)

T89_PARAMS = {
    0: {'C3': 1.813, 'a_T': 8.161, 'D': 2.08, 'x0': -8.799, 'Rs': 9.084},
    1: {'C3': 2.316, 'a_T': 8.119, 'D': 1.664, 'x0': -9.934, 'Rs': 9.238},
    2: {'C3': 2.641, 'a_T': 6.283, 'D': 1.541, 'x0': -4.183, 'Rs': 8.573},
    3: {'C3': 3.181, 'a_T': 6.266, 'D': 0.9351, 'x0': -5.389, 'Rs': 5.935},
    4: {'C3': 3.607, 'a_T': 6.196, 'D': 0.7677, 'x0': -5.072, 'Rs': 3.935},
    5: {'C3': 4.090, 'a_T': 5.831, 'D': 0.3325, 'x0': -6.472, 'Rs': 3.081},
}


def dipole_field(x, z, B0=31200.0):
    """Earth's dipole magnetic field in GSM coordinates (psi=0).

    Args:
        x: GSM x-coordinate (R_E).
        z: GSM z-coordinate (R_E).
        B0: Dipole moment (nT * R_E^3). Default: 31200 nT.

    Returns:
        Bx, Bz: Magnetic field components (nT).
    """
    r2 = x**2 + z**2
    r = np.sqrt(r2)
    r5 = r2 * r2 * r + 1e-30  # Avoid division by zero
    Bx = 3.0 * x * z * B0 / r5
    Bz = (3.0 * z**2 - r2) * B0 / r5
    return Bx, Bz


def tail_field(x, z, Kp_bin=2):
    """Simplified tail current sheet field in the noon-midnight plane (y=0).

    Uses a Harris-type current sheet model with Kp-dependent parameters.

    Args:
        x: GSM x-coordinate (R_E).
        z: GSM z-coordinate (R_E).
        Kp_bin: Kp level index (0-5).

    Returns:
        Bx_tail, Bz_tail: Tail field components (nT).
    """
    p = T89_PARAMS[Kp_bin]
    C3 = p['C3']
    a_T = p['a_T']
    D = p['D']
    x0 = p['x0']

    # Truncation factor: confines tail field to nightside
    W = 0.5 * (1.0 - (x - x0) / np.sqrt((x - x0)**2 + 100.0))

    # Finite-thickness current sheet contribution
    zeta = np.sqrt(z**2 + D**2)
    rho = np.abs(x)
    az = a_T + zeta
    r_term = np.sqrt(az**2 + rho**2)

    # Amplitude scaling (C3 * typical scale factor)
    amplitude = C3 * 10.0  # nT scale

    # Bx component: from -dA/dz, dominant near current sheet
    sign_z = np.where(np.abs(z) > 1e-10, np.sign(z), 0.0)
    Bx_tail = amplitude * W * z / (zeta * r_term) * sign_z

    # Bz component: from (1/rho) d(rho*A)/drho
    Bz_tail = -amplitude * W * D**2 / (zeta * r_term**3) * rho

    return Bx_tail, Bz_tail


def total_field(x, z, Kp_bin=2, B0=31200.0):
    """Total magnetic field: dipole + tail.

    Args:
        x: GSM x-coordinate (R_E).
        z: GSM z-coordinate (R_E).
        Kp_bin: Kp level index (0-5).
        B0: Dipole moment (nT * R_E^3).

    Returns:
        Bx, Bz: Total field components (nT).
    """
    r = np.sqrt(x**2 + z**2)
    # Only apply field outside Earth
    mask = r > 1.0

    Bx_dip, Bz_dip = dipole_field(x, z, B0)
    Bx_tail, Bz_tail = tail_field(x, z, Kp_bin)

    Bx = np.where(mask, Bx_dip + Bx_tail, 0.0)
    Bz = np.where(mask, Bz_dip + Bz_tail, 0.0)
    return Bx, Bz


print("Model functions defined / 모델 함수 정의 완료")
print(f"Available Kp bins: {list(T89_PARAMS.keys())}")
print(f"\nKp-dependent current sheet half-thickness D:")
for kp, p in T89_PARAMS.items():
    print(f"  Kp={kp}: D={p['D']:.3f} R_E")

## Part 4: Field Line Tracing / 자기장 선 추적

We trace field lines by integrating along the magnetic field direction. Starting from different latitudes on Earth's surface, we follow field lines outward to visualize the magnetospheric structure at different Kp levels (cf. Figs. 8–12 in the paper).

자기장 방향을 따라 적분하여 자기장 선을 추적합니다. 지구 표면의 서로 다른 위도에서 시작하여, 다양한 Kp 수준에서의 자기권 구조를 시각화합니다 (논문의 그림 8–12 참조).

In [ ]:
def trace_field_line(x0, z0, Kp_bin=2, ds=0.05, max_steps=20000, direction=1):
    """Trace a single field line from starting point (x0, z0).

    Args:
        x0, z0: Starting coordinates (R_E).
        Kp_bin: Kp level index.
        ds: Step size (R_E).
        max_steps: Maximum integration steps.
        direction: +1 (along B) or -1 (against B).

    Returns:
        xs, zs: Arrays of coordinates along the field line.
    """
    xs, zs = [x0], [z0]
    x, z = x0, z0

    for _ in range(max_steps):
        Bx, Bz = total_field(x, z, Kp_bin)
        B_mag = np.sqrt(Bx**2 + Bz**2)
        if B_mag < 1e-3:
            break

        # Normalized step
        dx = direction * Bx / B_mag * ds
        dz = direction * Bz / B_mag * ds

        x += dx
        z += dz
        r = np.sqrt(x**2 + z**2)

        # Stop conditions: hit Earth or go too far
        if r < 1.0 or r > 60.0 or np.abs(x) > 55.0:
            xs.append(x)
            zs.append(z)
            break

        xs.append(x)
        zs.append(z)

    return np.array(xs), np.array(zs)


def plot_field_lines(Kp_bin, ax, title=''):
    """Plot field lines for a given Kp level.

    Args:
        Kp_bin: Kp level index.
        ax: Matplotlib axes.
        title: Plot title.
    """
    # Starting latitudes (degrees from equator)
    latitudes = np.arange(60, 82, 2)

    for lat in latitudes:
        theta = np.radians(90 - lat)
        # Start from Earth's surface on the dayside (noon)
        x0_day = np.sin(theta)
        z0_day = np.cos(theta)
        xs_f, zs_f = trace_field_line(x0_day, z0_day, Kp_bin, ds=0.05, direction=1)
        xs_b, zs_b = trace_field_line(x0_day, z0_day, Kp_bin, ds=0.05, direction=-1)
        xs = np.concatenate([xs_b[::-1], xs_f[1:]])
        zs = np.concatenate([zs_b[::-1], zs_f[1:]])
        ax.plot(xs, zs, 'b-', linewidth=0.6, alpha=0.8)

        # Start from nightside
        x0_night = -np.sin(theta)
        z0_night = np.cos(theta)
        xs_f, zs_f = trace_field_line(x0_night, z0_night, Kp_bin, ds=0.05, direction=1)
        xs_b, zs_b = trace_field_line(x0_night, z0_night, Kp_bin, ds=0.05, direction=-1)
        xs = np.concatenate([xs_b[::-1], xs_f[1:]])
        zs = np.concatenate([zs_b[::-1], zs_f[1:]])
        ax.plot(xs, zs, 'b-', linewidth=0.6, alpha=0.8)

        # Southern hemisphere (mirror)
        x0_day_s = np.sin(theta)
        z0_day_s = -np.cos(theta)
        xs_f, zs_f = trace_field_line(x0_day_s, z0_day_s, Kp_bin, ds=0.05, direction=1)
        xs_b, zs_b = trace_field_line(x0_day_s, z0_day_s, Kp_bin, ds=0.05, direction=-1)
        xs = np.concatenate([xs_b[::-1], xs_f[1:]])
        zs = np.concatenate([zs_b[::-1], zs_f[1:]])
        ax.plot(xs, zs, 'b-', linewidth=0.6, alpha=0.8)

        x0_night_s = -np.sin(theta)
        z0_night_s = -np.cos(theta)
        xs_f, zs_f = trace_field_line(x0_night_s, z0_night_s, Kp_bin, ds=0.05, direction=1)
        xs_b, zs_b = trace_field_line(x0_night_s, z0_night_s, Kp_bin, ds=0.05, direction=-1)
        xs = np.concatenate([xs_b[::-1], xs_f[1:]])
        zs = np.concatenate([zs_b[::-1], zs_f[1:]])
        ax.plot(xs, zs, 'b-', linewidth=0.6, alpha=0.8)

    # Draw Earth
    earth = Circle((0, 0), 1.0, color='steelblue', zorder=5)
    ax.add_patch(earth)
    ax.set_xlim(15, -50)
    ax.set_ylim(-20, 20)
    ax.set_xlabel(r'$X_{GSM}$ ($R_E$)')
    ax.set_ylabel(r'$Z_{GSM}$ ($R_E$)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color='gray', linewidth=0.3)
    ax.axvline(0, color='gray', linewidth=0.3)

In [ ]:
# Reproduce Figs. 8-11: Field line patterns for different Kp levels
# 그림 8-11 재현: 다양한 Kp 수준에서의 자기장 선 패턴

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

kp_configs = [
    (0, r'Kp = 0, 0$^+$ (very quiet / 매우 조용)'),
    (2, r'Kp = 2$^-$, 2, 2$^+$ (average quiet / 평균적 조용)'),
    (4, r'Kp = 4$^-$, 4, 4$^+$ (disturbed / 교란)'),
    (5, r'Kp $\geq$ 5$^-$ (strongly disturbed / 강한 교란)'),
]

for (kp_bin, title), ax in zip(kp_configs, axes.flat):
    plot_field_lines(kp_bin, ax, title)

fig.suptitle('Field Line Patterns in Noon-Midnight Meridian Plane (cf. Figs. 8–11)\n'
             'Noon-Midnight 자오면에서의 자기장 선 패턴',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: Kp Dependence of Key Parameters / Kp에 따른 핵심 매개변수 변화

Visualize how key model parameters change with Kp index, as discussed in Section 5.1 of the paper (Table 1).

논문 Section 5.1(Table 1)에서 논의된 것처럼, Kp 지수에 따른 핵심 모델 매개변수의 변화를 시각화합니다.

In [ ]:
# Plot Kp dependence of key parameters from Table 1
# Table 1에서 핵심 매개변수의 Kp 의존성 시각화

kp_vals = list(T89_PARAMS.keys())
C3_vals = [T89_PARAMS[k]['C3'] for k in kp_vals]
D_vals = [T89_PARAMS[k]['D'] for k in kp_vals]
aT_vals = [T89_PARAMS[k]['a_T'] for k in kp_vals]
Rs_vals = [T89_PARAMS[k]['Rs'] for k in kp_vals]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# C3: tail current amplitude
axes[0, 0].plot(kp_vals, C3_vals, 'ro-', markersize=8, linewidth=2)
axes[0, 0].set_xlabel('Kp bin')
axes[0, 0].set_ylabel(r'$C_3$')
axes[0, 0].set_title(r'$C_3$ — Tail Current Amplitude / 꼬리 전류 진폭')
axes[0, 0].grid(True, alpha=0.3)

# D: current sheet half-thickness
axes[0, 1].plot(kp_vals, D_vals, 'bs-', markersize=8, linewidth=2)
axes[0, 1].set_xlabel('Kp bin')
axes[0, 1].set_ylabel(r'$D$ ($R_E$)')
axes[0, 1].set_title(r'$D$ — Current Sheet Half-Thickness / 전류판 반두께')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].annotate('Thinner = more intense\n더 얇음 = 더 강한 전류',
                     xy=(4, 0.5), fontsize=10, color='blue',
                     ha='center')

# a_T: radial scale length
axes[1, 0].plot(kp_vals, aT_vals, 'g^-', markersize=8, linewidth=2)
axes[1, 0].set_xlabel('Kp bin')
axes[1, 0].set_ylabel(r'$a_T$ ($R_E$)')
axes[1, 0].set_title(r'$a_T$ — Radial Scale Length / 방사형 스케일 길이')
axes[1, 0].grid(True, alpha=0.3)

# Rs: hinging distance
axes[1, 1].plot(kp_vals, Rs_vals, 'mD-', markersize=8, linewidth=2)
axes[1, 1].set_xlabel('Kp bin')
axes[1, 1].set_ylabel(r'$R_s$ ($R_E$)')
axes[1, 1].set_title(r'$R_s$ — Hinging Distance / 힌지 거리')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Kp Dependence of T89 Model Parameters (Table 1)\n'
             'T89 모델 매개변수의 Kp 의존성',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observations / 핵심 관찰:")
print(f"  C3 increases {C3_vals[-1]/C3_vals[0]:.1f}x from Kp=0 to Kp≥5 (tail strengthens)")
print(f"  D decreases {D_vals[0]/D_vals[-1]:.1f}x from Kp=0 to Kp≥5 (sheet thins)")
print(f"  Rs decreases {Rs_vals[0]/Rs_vals[-1]:.1f}x from Kp=0 to Kp≥5 (hinge moves earthward)")

## Summary / 요약

| Concept / 개념 | This Paper (T89) / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Magnetic field model / 자기장 모델 | Empirical, modular current systems / 경험적, 모듈식 전류계 | T96, TS04, TS07 — more parameters, time history / 더 많은 매개변수, 시간 이력 |
| Input parameter / 입력 매개변수 | Kp index only / Kp 지수만 | Solar wind $P_{dyn}$, IMF $B_y$, $B_z$, Dst / 태양풍 직접 매개변수 |
| Tail current sheet / 꼬리 전류판 | Warped by dipole tilt (eq. 11) / dipole tilt에 의한 warping | Similar warping + storm-time deformation / 유사한 warping + 폭풍 시 변형 |
| Data basis / 데이터 기반 | 36,682 vectors (IMP, HEOS, 1966–1980) | Millions of vectors (THEMIS, MMS, etc.) / 수백만 벡터 |
| Implementation / 구현 | Custom Fortran code | `geopack` (Fortran/Python), IRBEM, SpacePy |
| Field line tracing / 자기장 선 추적 | Manual integration / 수동 적분 | `geopack.trace()`, IRBEM tracing / 자동화된 추적 도구 |

### Key insight from implementation / 구현으로부터의 핵심 통찰

The field line plots clearly show how increasing Kp stretches nightside field lines dramatically while dayside field lines remain relatively unchanged. This asymmetric response is the physical signature of the tail current sheet intensification during geomagnetic storms.

Field line 그림은 Kp가 증가함에 따라 nightside 자기장 선이 극적으로 늘어나는 반면 dayside 자기장 선은 비교적 변하지 않음을 명확히 보여줍니다. 이 비대칭적 반응은 지자기 폭풍 동안 꼬리 전류판 강화의 물리적 서명입니다.